In [ ]:
"""
* 주제 : 와인종류 분류하기
* 사용데이터 : 08_wine_exe.csv
  - 전처리 필요함
 
<분석방법>
 - 상관관계 분석..
 - 스케일링 : 모든 스케일러 각각 사용
 - 데이터분류 : 훈련:검증:테스트 = 6 : 2 : 2로 분류하여 사용
 - 튜닝 없이 전체 모델 훈련하여 ~ 평가까지..
 - 하이퍼파라메터 튜닝 후 전체 모델 훈련하여 ~ 평가까지..
 - 이외 훈련에 영햘을 미칠 수 있는, 모든 경우의 수 적용


<튜닝 클래스 사용 및 속성 : 제공된 값 이외에 각자 임의의 값 추가 적용하여 성능 테스트하기>
 * 랜덤포레스트, 엑스트라트리, 그레디언트부스트
  - n_estimators = [50, 100]
    : 트리 갯수 지정(보통 50~1000 사이값 사용)
  - max_depth = [None, 10]
    : 트리가 뻗어나가는 최대깊이 지정(None은 제한없음, 보통 3~10 사이값 사용)
  - min_samples_split = [2, 5]
    : 노드를 분할하기 위한 최소한의 샘플 수 (보통 2~10 사이값 사용)
  - min_samples_leaf = [1, 2, 4]
    : 리프노드(결정노드)의 최소한의 샘플 수 (보통 1~10 사이값 사용)
   
 * 히스트그레디언트부스트
  - max_iter = [50, 100]
    : 트리 갯수 지정(보통 50~1000 사이값 사용)
  - max_depth = [None, 10]
    : 트리가 뻗어나가는 최대깊이 지정(None은 제한없음, 보통 3~10 사이값 사용)
  - min_samples_leaf = [1, 2, 4]
    : 리프노드(결정노드)의 최소한의 샘플 수 (보통 1~10 사이값 사용)
   
 * 엑스지부스트
  - n_estimators = [50, 100]
    : 트리 갯수 지정(보통 50~1000 사이값 사용)
  - max_depth = [None, 10]
    : 트리가 뻗어나가는 최대깊이 지정(None은 제한없음, 보통 3~10 사이값 사용)
  - min_child_weight = [1, 2, 4]
    : 리프노드(결정노드)의 최소한의 샘플 수 (보통 1~10 사이값 사용)


<튜닝 자동화 모델(GridSearchCV 클래스)에서 사용할 튜닝 속성>
 * scoring = accuracy
    : 튜닝 자동화 모델(클래스)에 사용할 평가 방법 지정
    : 분류에서는 정확도를 이용
 
 * refit = accuracy
    : 튜닝 자동화 모델(클래스)에 사용할 모델 선정 기준 지정
    : scoring에서 한가지만 사용되기에 생략가능
 
 * cv = 5
    : 튜닝 자동화 모델(클래스)에 사용할 "교차검증" 시 사용할 Fold 갯수
    : Fold(폴드) -> 모델 훈련시 사용할 훈련데이터를 내부적으로
                 -> 폴드의 갯수만큼 다시 분류(훈련:검증으로)하여 사용하는 방법
                 
<최종 결과>
 * 각 처리 진행 시 해석 주석으로 작성
 
 * 데이터프레임에 아래 항목 저장하여 비교 후 모델 선정 (튜닝 전과 후로 각각 진행)
  - 모델명, 경우의수, 훈련정확도, 검증정확도, 테스트정확도,
    (훈련-검증)정확도, 정밀도, 재현율, F1-Score


 * 모델 선정 후, 테스트 데이터로 최종 예측 진행 (최종 선정 모델로 한번 진행)

"""

In [ ]:
### 사용할 라이브러리
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
plt.rcParams["figure.figsize"] = (12, 5)
plt.rcParams["font.family"] = "Malgun Gothic"
plt.rcParams["axes.unicode_minus"] = False
from scipy.stats import spearmanr, pearsonr
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GridSearchCV
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.ensemble import HistGradientBoostingClassifier
from xgboost import XGBRegressor
from xgboost import XGBClassifier
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score


In [ ]:
# 파일 불러오기
file_path = '../data/08_wine_exe.csv'
wine = pd.read_csv(file_path)
# 결측치 및 이상치 제거
wine = wine.dropna().reset_index(drop=True)
wine = wine[wine["alcohol"] < 100].reset_index(drop=True)
wine = wine[wine["sugar"] < 100].reset_index(drop=True)

In [ ]:
# 상관관계 분석
corr = wine.corr()
X = wine[["alcohol", "sugar", "pH"]]
y = wine["class"]

for col in X.columns:
    pearson_corr, pearson_p = pearsonr(X[col], y)
    spearman_corr, spearman_p = spearmanr(X[col], y)

    print(f"{col}과 와인 종류의 상관관계")
    print("Pearson :", round(pearson_corr, 4), ", p-value :", round(pearson_p, 6))
    print("Spearman :", round(spearman_corr, 4), ", p-value :", round(spearman_p, 6))


In [ ]:
# 데이터 분류 6 : 2 : 2
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.4, random_state=42, stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp
)
print("훈련 :", X_train.shape)
print("검증 :", X_val.shape)
print("테스트 :", X_test.shape)


In [ ]:
# 스케일링
scaled =  StandardScaler()

train_scaled = scaled.fit_transform(X_train)
val_scaled = scaled.transform(X_val)
test_scaled = scaled.transform(X_test)

print("훈련 :", train_scaled.shape)
print("검증 :", val_scaled.shape)
print("테스트 :", test_scaled.shape)


In [ ]:
#튜닝 없이 전체 모델 훈련
model_list = [
    RandomForestClassifier(random_state=42, n_jobs=-1),
    ExtraTreesClassifier(random_state=42, n_jobs=-1),
    GradientBoostingClassifier(random_state=42),
    HistGradientBoostingClassifier(random_state=42),
    XGBClassifier(random_state=42, n_jobs=-1)
]
best_model = []
for model in model_list:
    model_name = model.__class__.__name__

    model.fit(train_scaled, y_train)

    train_pred = model.predict(train_scaled)
    val_pred = model.predict(val_scaled)
    test_pred = model.predict(test_scaled)

    #훈련(정확도, 정밀도, 재현율, f1-score)
    train_acc = accuracy_score(y_train, train_pred)
    train_pre = precision_score(y_train, train_pred)
    train_rec = recall_score(y_train, train_pred)
    train_f1 = f1_score(y_train, train_pred)

    # 검증(정확도, 정밀도, 재현율, f1-score)
    val_acc = accuracy_score(y_val, val_pred)
    val_pre = precision_score(y_val, val_pred)
    val_rec = recall_score(y_val, val_pred)
    val_f1 = f1_score(y_val, val_pred)

    # 테스트(정확도, 정밀도, 재현율, f1-score)
    test_acc = accuracy_score(y_test, test_pred)
    test_pre = precision_score(y_test, test_pred)
    test_rec = recall_score(y_test, test_pred)
    test_f1 = f1_score(y_test, test_pred)
    
    best_model.append([model_name, train_acc, val_acc, test_acc, train_acc - val_acc, train_pre - val_pre, train_rec - val_rec, train_f1 - val_f1])

    print(f"{model_name}모델의 훈련 정확도 : {train_acc:.5f}, 검증 정확도 : {val_acc:.5f}, 테스트 정확도 : {test_acc:.5f}")
    print(f"(훈련-검증) 정확도 : {train_acc - val_acc:.5f}, (훈련-검증) 정밀도 : {train_pre - val_pre:.5f}, (훈련-검증) 재현율 : {train_rec - val_rec:.5f}, (훈련-검증) F1-score : {train_f1 - val_f1:.5f}")

In [ ]:
best_df = pd.DataFrame(
    best_model,
    columns=["모델명", "훈련정확도", "검증정확도", "테스트정확도",
            "훈련검증정확도", "훈련검증정밀도", "훈련검증재현율", "훈련검증F1"]
)
best_result = best_df.loc[best_df["훈련검증정확도"].idxmin()]
print("")
print("최적 모델 :", best_result["모델명"])
print("훈련 정확도 :", best_result["훈련정확도"])
print("검증 정확도 :", best_result["검증정확도"])
print("테스트 정확도 :", best_result["테스트정확도"])
print("(훈련 - 검증)정확도 :", best_result["훈련검증정확도"])
print("(훈련 - 검증)정밀도 :", best_result["훈련검증정밀도"])
print("(훈련 - 검증)재현율 :", best_result["훈련검증재현율"])
print("(훈련 - 검증)F1 :", best_result["훈련검증F1"])


In [ ]:
best = []

for model in model_list:
    model_name = model.__class__.__name__

    if model_name == "RandomForestClassifier" or model_name == "ExtraTreesClassifier" or model_name == "GradientBoostingClassifier":
        gridParams = {
            "n_estimators": [50, 100, 150],
            "max_depth": [None, 10],
            "min_samples_split": [2, 5],
            "min_samples_leaf": [1, 2, 4]
        }

    elif model_name == "HistGradientBoostingClassifier":
        gridParams = {
            "max_iter": [50, 100, 150],
            "max_depth": [None, 10],
            "min_samples_leaf": [1, 2, 4]
        }

    elif model_name == "XGBClassifier":
        gridParams = {
            "n_estimators": [50, 100, 150],
            "max_depth": [None, 10],
            "min_child_weight": [1, 2, 4]
        }

    grid_search_model = GridSearchCV(
        estimator=model,
        param_grid=gridParams,
        scoring="accuracy",
        cv=5,
        n_jobs=-1
    )
    grid_search_model.fit(train_scaled, y_train)

    best_model = grid_search_model.best_estimator_
    best_params = grid_search_model.best_params_

    train_pred = best_model.predict(train_scaled)
    val_pred = best_model.predict(val_scaled)
    test_pred = best_model.predict(test_scaled)

    #훈련(정확도, 정밀도, 재현율, f1-score)
    train_acc = accuracy_score(y_train, train_pred)
    train_pre = precision_score(y_train, train_pred)
    train_rec = recall_score(y_train, train_pred)
    train_f1 = f1_score(y_train, train_pred)

    # 검증(정확도, 정밀도, 재현율, f1-score)
    val_acc = accuracy_score(y_val, val_pred)
    val_pre = precision_score(y_val, val_pred)
    val_rec = recall_score(y_val, val_pred)
    val_f1 = f1_score(y_val, val_pred)

    # 테스트(정확도, 정밀도, 재현율, f1-score)
    test_acc = accuracy_score(y_test, test_pred)
    test_pre = precision_score(y_test, test_pred)
    test_rec = recall_score(y_test, test_pred)
    test_f1 = f1_score(y_test, test_pred)

    best.append([model_name, train_acc, val_acc, test_acc, train_acc - val_acc, train_pre - val_pre, train_rec - val_rec, train_f1 - val_f1])

    

In [ ]:
best_df = pd.DataFrame(
    best,
    columns=["모델명", "훈련정확도", "검증정확도", "테스트정확도",
            "훈련검증정확도", "훈련검증정밀도", "훈련검증재현율", "훈련검증F1"]
)
best_result = best_df.loc[best_df["훈련검증정확도"].idxmin()]
print("")
print("최적 모델 :", best_result["모델명"])
print("훈련 정확도 :", best_result["훈련정확도"])
print("검증 정확도 :", best_result["검증정확도"])
print("테스트 정확도 :", best_result["테스트정확도"])
print("(훈련 - 검증)정확도 :", best_result["훈련검증정확도"])
print("(훈련 - 검증)정밀도 :", best_result["훈련검증정밀도"])
print("(훈련 - 검증)재현율 :", best_result["훈련검증재현율"])
print("(훈련 - 검증)F1 :", best_result["훈련검증F1"])